In [1]:
# High-performance VAE imputation for H and V
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers, optimizers, regularizers, initializers, callbacks
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, QuantileTransformer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy import stats
from scipy import special
import warnings
import random
import os
import gc
import time

warnings.filterwarnings("ignore")

# Check the TensorFlow version
print(f"TensorFlow version: {tf.__version__}")

# ==================== Enhanced reproducibility settings ====================
SEED = 42

def set_all_seeds(seed=42):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# Initialize random seeds
set_all_seeds(SEED)

# GPU memory settings
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Found {len(gpus)} GPU devices")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")
else:
    print("No GPU found. Running on the CPU.")

# Clear the session
tf.keras.backend.clear_session()
gc.collect()

# Implement the inverse Box-Cox transform
def inv_boxcox(y, lambda_param):
    """Apply the inverse Box-Cox transform."""
    if lambda_param == 0:
        return np.exp(y)
    else:
        # Avoid invalid results from negative values
        return np.power(lambda_param * y + 1, 1 / lambda_param)

# 1. Data loading and preprocessing
def load_and_preprocess_data():
    """Load the data and perform initial preprocessing."""
    data = pd.read_csv("./Others.csv", low_memory=True)
    print(f"Original data shape: {data.shape}")
    
    # Keep a copy of the original data
    original_data = data.copy()
    
    # Drop nonnumeric columns
    cols_to_drop = []
    for col in data.columns:
        try:
            pd.to_numeric(data[col].dropna().iloc[:10])
        except:
            cols_to_drop.append(col)
    
    data = data.drop(columns=cols_to_drop)
    
    # Drop ID columns
    for col in data.columns[:3]:
        if data[col].dtype in ['int64', 'float64']:
            non_null = data[col].dropna()
            if len(non_null) > 1:
                diffs = non_null.diff().dropna()
                if (diffs == 1).mean() > 0.9:
                    data = data.drop(columns=[col])
    
    # Drop the L(m) column
    if 'L(m)' in data.columns:
        data = data.drop(columns=['L(m)'])
    
    # Drop columns with more than 30% missing values
    missing_ratio = data.isnull().sum() / len(data)
    cols_to_drop = missing_ratio[missing_ratio > 0.30].index
    data = data.drop(columns=cols_to_drop)
    
    # Convert data types
    for col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce').astype('float32')
    
    print(f"Final data shape: {data.shape}")
    print(f"Features: {list(data.columns)}")
    
    # Check whether H and V are the only features
    if len(data.columns) == 2:
        print(f"Confirmed {len(data.columns)} features")
    
    return data, original_data

# 2. Outlier treatment for H and V
def enhanced_outlier_handling(df):
    """Apply feature-specific outlier treatment to H and V."""
    df_clean = df.copy()
    transform_info = {}
    
    for column in df_clean.columns:
        mask = df_clean[column].notna()
        values = df_clean.loc[mask, column].values
        
        if len(values) == 0:
            continue
        
        # H: Box-Cox transform with IQR clipping
        if 'H' in column:
            print(f"{column}: applying Box-Cox and IQR clipping")
            
            # Ensure the data are positive
            min_val = values.min()
            if min_val <= 0:
                shift = 1 - min_val
                values = values + shift
            else:
                shift = 0
            
            # Box-Cox transform
            values_transformed, lambda_param = stats.boxcox(values)
            
            # IQR outlier treatment
            q1, q3 = np.percentile(values_transformed, [25, 75])
            iqr = q3 - q1
            lower = q1 - 3 * iqr  # Use a moderate IQR multiplier of 3
            upper = q3 + 3 * iqr
            values_transformed = np.clip(values_transformed, lower, upper)
            
            transform_info[column] = {'type': 'boxcox', 'lambda': lambda_param, 'shift': shift}
            df_clean[column] = df_clean[column].astype(values_transformed.dtype, copy=False)
            df_clean.loc[mask, column] = values_transformed
            print(f"  Box-Cox lambda={lambda_param:.4f}, shift={shift:.4f}")
            print(f"  IQR clipping range: [{lower:.2f}, {upper:.2f}]")
            
        # V: quantile transform
        elif 'V' in column:
            print(f"{column}: applying a quantile transform to a highly skewed feature")
            
            # Apply a quantile transform to the highly skewed feature
            qt = QuantileTransformer(
                n_quantiles=min(1000, len(values)), 
                output_distribution='normal', 
                random_state=SEED
            )
            values_transformed = qt.fit_transform(values.reshape(-1, 1)).flatten()
            
            transform_info[column] = {'type': 'quantile', 'transformer': qt}
            df_clean[column] = df_clean[column].astype(values_transformed.dtype, copy=False)
            df_clean.loc[mask, column] = values_transformed
            print(f"  Quantile transform complete. Output follows a standard normal distribution.")
            
        else:
            # Apply standard IQR clipping to any other features
            print(f"{column}: applying standard IQR outlier treatment")
            
            q1, q3 = np.percentile(values, [25, 75])
            iqr = q3 - q1
            lower = q1 - 2.5 * iqr
            upper = q3 + 2.5 * iqr
            values = np.clip(values, lower, upper)
            
            transform_info[column] = {'type': 'clip', 'shift': 0}
            df_clean[column] = df_clean[column].astype(values.dtype, copy=False)
            df_clean.loc[mask, column] = values
    
    return df_clean, transform_info

# 3. Scaling for H and V
def enhanced_normalization(data):
    """Scale H and V using feature-specific methods."""
    data_normalized = pd.DataFrame(index=data.index, columns=data.columns)
    scalers = {}
    
    for column in data.columns:
        mask = data[column].notna()
        values = data[column][mask].values.reshape(-1, 1)
        
        if len(values) > 0:
            # Choose the scaler by feature
            if 'V' in column:
                # Scale V with StandardScaler after the quantile transform
                scaler = StandardScaler()
                print(f"{column}: using StandardScaler")
            elif 'H' in column:
                # Scale H with RobustScaler after the Box-Cox transform
                scaler = RobustScaler(quantile_range=(10, 90))
                print(f"{column}: using RobustScaler")
            else:
                # Use RobustScaler by default
                scaler = RobustScaler(quantile_range=(5, 95))
            
            scaler.fit(values)
            scaled = scaler.transform(values).flatten()
            
            # Map values to [0.1, 0.9] to avoid edge effects
            min_val, max_val = scaled.min(), scaled.max()
            if max_val - min_val > 1e-8:
                scaled = (scaled - min_val) / (max_val - min_val) * 0.8 + 0.1
            else:
                scaled = np.ones_like(scaled) * 0.5
            
            data_normalized.loc[mask, column] = scaled.astype(np.float32)
            scalers[column] = {'scaler': scaler, 'min': min_val, 'max': max_val}
    
    return data_normalized, scalers

# Inverse scaling
def inverse_transform_array(data_normalized_array, scalers, transform_info, original_columns, original_index=None):
    """Convert the scaled NumPy array to a DataFrame in the original units."""
    n_samples = data_normalized_array.shape[0]
    
    if original_index is None:
        original_index = range(n_samples)
    
    data_original = pd.DataFrame(index=original_index, columns=original_columns)
    
    for i, column in enumerate(original_columns):
        if column not in scalers:
            data_original[column] = data_normalized_array[:, i]
            continue
            
        values = data_normalized_array[:, i].copy()
        
        # Reverse the scaling
        scaler_info = scalers[column]
        min_val = scaler_info['min']
        max_val = scaler_info['max']
        
        # Map [0.1, 0.9] back to the original range
        values = (values - 0.1) / 0.8 * (max_val - min_val) + min_val
        
        # Invert the scaler
        values = scaler_info['scaler'].inverse_transform(values.reshape(-1, 1)).flatten()
        
        # Invert the transform
        if column in transform_info:
            trans_type = transform_info[column]['type']
            
            if trans_type == 'quantile':
                # Invert the quantile transform
                transformer = transform_info[column]['transformer']
                values = transformer.inverse_transform(values.reshape(-1, 1)).flatten()
            elif trans_type == 'boxcox':
                # Invert the Box-Cox transform using the helper above
                lambda_param = transform_info[column]['lambda']
                shift = transform_info[column]['shift']
                values = inv_boxcox(values, lambda_param) - shift
        
        data_original[column] = values
    
    return data_original

# 4. Enhanced VAE for two features
class EnhancedVAE:
    def __init__(self, data_dim, latent_dim=4, learning_rate=0.0005, seed=42):
        """Enhanced VAE optimized for two features."""
        self.data_dim = data_dim
        self.latent_dim = latent_dim  # Use a smaller latent dimension for two features
        self.learning_rate = learning_rate
        self.seed = seed
        
        tf.random.set_seed(seed)
        
        # Build the enhanced model
        self.encoder = self._build_encoder()
        self.decoder = self._build_decoder()
        
        # Use the Adam optimizer
        self.optimizer = optimizers.Adam(learning_rate=learning_rate)
        
        self.best_weights = None
        self.best_val_loss = float('inf')
    
    def _build_encoder(self):
        """Build the encoder for two features."""
        inputs = layers.Input(shape=(self.data_dim,))
        
        # Network architecture for two features
        x = layers.Dense(32, kernel_initializer='he_normal')(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.2)(x)
        
        x = layers.Dense(16, kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.15)(x)
        
        x = layers.Dense(8, kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        
        z_mean = layers.Dense(self.latent_dim)(x)
        z_log_var = layers.Dense(self.latent_dim)(x)
        
        # Limit the range of log_var
        z_log_var = layers.Lambda(lambda x: tf.clip_by_value(x, -3, 3))(z_log_var)
        
        return keras.Model(inputs, [z_mean, z_log_var])
    
    def _build_decoder(self):
        """Build the decoder for two features."""
        inputs = layers.Input(shape=(self.latent_dim,))
        
        x = layers.Dense(8, kernel_initializer='he_normal')(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        
        x = layers.Dense(16, kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.15)(x)
        
        x = layers.Dense(32, kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.2)(x)
        
        outputs = layers.Dense(self.data_dim, activation='sigmoid')(x)
        
        return keras.Model(inputs, outputs)
    
    def _sample(self, z_mean, z_log_var):
        """Sample using the reparameterization trick."""
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon * 0.05  # Reduce noise
    
    @tf.function
    def train_step(self, x_batch, mask_batch, kl_weight=0.0005):
        """Run one training step."""
        with tf.GradientTape() as tape:
            # Encode
            z_mean, z_log_var = self.encoder(x_batch, training=True)
            z = self._sample(z_mean, z_log_var)
            
            # Decode
            reconstructed = self.decoder(z, training=True)
            
            # Compute reconstruction loss on observed values
            mse = tf.square(x_batch - reconstructed) * mask_batch
            reconstruction_loss = tf.reduce_sum(mse) / tf.reduce_sum(mask_batch)
            
            # KL loss
            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
            )
            
            # Total loss
            total_loss = reconstruction_loss + kl_weight * kl_loss
        
        # Update gradients
        variables = self.encoder.trainable_variables + self.decoder.trainable_variables
        gradients = tape.gradient(total_loss, variables)
        gradients = [tf.clip_by_norm(g, 1.0) for g in gradients]
        self.optimizer.apply_gradients(zip(gradients, variables))
        
        return total_loss
    
    def fit(self, data, mask, validation_split=0.2, epochs=500, batch_size=64):
        """Train the model."""
        # Split the data
        n_samples = len(data)
        indices = np.arange(n_samples)
        np.random.seed(self.seed)
        np.random.shuffle(indices)
        
        split_point = int(n_samples * (1 - validation_split))
        train_indices = indices[:split_point]
        val_indices = indices[split_point:]
        
        x_train = data[train_indices]
        x_val = data[val_indices]
        mask_train = mask[train_indices]
        mask_val = mask[val_indices]
        
        print(f"Training set: {len(x_train)} samples")
        print(f"Validation set: {len(x_val)} samples")
        
        # Early-stopping settings
        patience = 50
        patience_counter = 0
        
        # Learning-rate schedule
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            self.learning_rate,
            decay_steps=1000,
            decay_rate=0.96,
            staircase=True
        )
        self.optimizer.learning_rate = lr_schedule
        
        for epoch in range(epochs):
            # Adjust the KL weight during training
            kl_weight = min(0.001, 0.0001 * (epoch / 100))
            
            # Train
            train_indices_shuffled = np.random.permutation(len(x_train))
            train_loss = 0
            n_batches = 0
            
            for i in range(0, len(x_train), batch_size):
                batch_indices = train_indices_shuffled[i:i+batch_size]
                loss = self.train_step(
                    x_train[batch_indices],
                    mask_train[batch_indices],
                    kl_weight
                )
                train_loss += loss
                n_batches += 1
            
            train_loss /= n_batches
            
            # Validation
            if epoch % 10 == 0:
                val_loss = 0
                n_val_batches = 0
                
                for i in range(0, len(x_val), batch_size):
                    z_mean, z_log_var = self.encoder(x_val[i:i+batch_size], training=False)
                    z = self._sample(z_mean, z_log_var)
                    reconstructed = self.decoder(z, training=False)
                    
                    mse = tf.square(x_val[i:i+batch_size] - reconstructed) * mask_val[i:i+batch_size]
                    val_loss += tf.reduce_sum(mse) / tf.reduce_sum(mask_val[i:i+batch_size])
                    n_val_batches += 1
                
                val_loss /= n_val_batches
                
                if val_loss < self.best_val_loss:
                    self.best_val_loss = val_loss
                    self.best_weights = {
                        'encoder': self.encoder.get_weights(),
                        'decoder': self.decoder.get_weights()
                    }
                    patience_counter = 0
                else:
                    patience_counter += 10
                
                if epoch % 50 == 0:
                    print(f"Epoch {epoch}: Train Loss={train_loss:.6f}, Val Loss={val_loss:.6f}")
                
                # Early stopping
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch}")
                    break
        
        # Restore the best weights
        if self.best_weights:
            self.encoder.set_weights(self.best_weights['encoder'])
            self.decoder.set_weights(self.best_weights['decoder'])
    
    def impute(self, data, mask):
        """Impute missing values."""
        imputed = np.zeros_like(data)
        batch_size = 128
        
        for i in range(0, len(data), batch_size):
            batch_data = data[i:i+batch_size]
            batch_mask = mask[i:i+batch_size]
            
            # Average multiple draws
            predictions = []
            for _ in range(20):  # Increase the number of draws
                z_mean, z_log_var = self.encoder(batch_data, training=False)
                z = self._sample(z_mean, z_log_var)
                pred = self.decoder(z, training=False).numpy()
                predictions.append(pred)
            
            pred_mean = np.mean(predictions, axis=0)
            
            # Combine observed and predicted values
            imputed[i:i+batch_size] = batch_data * batch_mask + pred_mean * (1 - batch_mask)
        
        return imputed

# 5. Simplified feature-specific deep learning models
class FeatureSpecificDeepModels:
    def __init__(self, seed=42):
        self.seed = seed
        self.models = {}
        self.scalers = {}
    
    def _build_feature_model(self, input_dim):
        """Build a simplified deep learning model for one feature."""
        model = keras.Sequential([
            layers.Dense(16, activation='relu', input_shape=(input_dim,)),
            layers.Dropout(0.2),
            layers.Dense(8, activation='relu'),
            layers.Dropout(0.1),
            layers.Dense(4, activation='relu'),
            layers.Dense(1)
        ])
        model.compile(optimizer='adam', loss='mse')
        return model
    
    def fit(self, data, mask, column_names=None):
        """Train a separate deep learning model for each feature."""
        print("Training feature-specific deep learning models...")
        
        for col_idx in range(data.shape[1]):
            col_name = column_names[col_idx] if column_names is not None else f"feature_{col_idx}"
            obs_mask = mask[:, col_idx] == 1
            
            if np.sum(obs_mask) < 100:
                continue
            
            # Prepare the training data using the other feature as input
            X_train = np.delete(data[obs_mask], col_idx, axis=1)
            y_train = data[obs_mask, col_idx]
            
            # Scale the data
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
            X_train_scaled = scaler_x.fit_transform(X_train.reshape(-1, 1))  # One input feature
            y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
            
            # Build and train the model
            model = self._build_feature_model(1)  # Input dimension of one
            
            # Early-stopping callback
            early_stop = callbacks.EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
            
            model.fit(
                X_train_scaled, y_train_scaled,
                epochs=100,
                batch_size=32,
                verbose=0,
                callbacks=[early_stop]
            )
            
            self.models[col_idx] = model
            self.scalers[col_idx] = {'x': scaler_x, 'y': scaler_y}
            print(f"  Finished training the model for {col_name}")
    
    def impute(self, data, mask):
        """Impute missing values."""
        imputed = data.copy()
        
        for col_idx, model in self.models.items():
            miss_mask = mask[:, col_idx] == 0
            if np.sum(miss_mask) == 0:
                continue
            
            X_miss = np.delete(imputed[miss_mask], col_idx, axis=1)
            
            # Scale the data
            X_miss_scaled = self.scalers[col_idx]['x'].transform(X_miss.reshape(-1, 1))
            
            # Predict
            pred_scaled = model.predict(X_miss_scaled, verbose=0).flatten()
            
            # Reverse the scaling
            pred = self.scalers[col_idx]['y'].inverse_transform(pred_scaled.reshape(-1, 1)).flatten()
            
            imputed[miss_mask, col_idx] = pred
        
        return imputed

# 6. Ensemble for H and V
class EnhancedEnsemble:
    def __init__(self, data_dim, seed=42):
        self.data_dim = data_dim
        self.seed = seed
        self.models = []
        self.weights = []
        self.feature_weights = {}  # Feature-specific weights
    
    def fit(self, data, mask, column_names):
        """Train the ensemble for H and V."""
        print("\nTraining the enhanced ensemble...")
        
        # 1. KNN (fast baseline)
        print("1. KNN imputer...")
        knn = KNNImputer(n_neighbors=15, weights='distance')
        knn.fit(data)
        self.models.append(('knn', knn))
        
        # 2. IterativeImputer with gradient boosting
        print("2. GradientBoosting IterativeImputer...")
        gb_mice = IterativeImputer(
            estimator=GradientBoostingRegressor(
                n_estimators=150,
                max_depth=5,
                learning_rate=0.1,
                subsample=0.8,
                random_state=self.seed
            ),
            max_iter=25,
            random_state=self.seed
        )
        gb_mice.fit(data)
        self.models.append(('gb_mice', gb_mice))
        
        # 3. IterativeImputer with a random forest
        print("3. IterativeImputer with a random forest...")
        rf_mice = IterativeImputer(
            estimator=RandomForestRegressor(
                n_estimators=150,
                max_depth=8,
                min_samples_split=3,
                min_samples_leaf=2,
                random_state=self.seed
            ),
            max_iter=20,
            random_state=self.seed
        )
        rf_mice.fit(data)
        self.models.append(('rf_mice', rf_mice))
        
        # 4. ExtraTrees IterativeImputer
        print("4. ExtraTrees IterativeImputer...")
        et_mice = IterativeImputer(
            estimator=ExtraTreesRegressor(
                n_estimators=150,
                max_depth=8,
                min_samples_split=3,
                random_state=self.seed
            ),
            max_iter=20,
            random_state=self.seed
        )
        et_mice.fit(data)
        self.models.append(('et_mice', et_mice))
        
        # 5. Feature-specific deep learning models
        print("5. Feature-specific deep learning models...")
        fs_model = FeatureSpecificDeepModels(seed=self.seed)
        fs_model.fit(data, mask, column_names)
        self.models.append(('feature_deep', fs_model))
        
        # 6. Enhanced VAE
        print("6. Enhanced VAE...")
        vae = EnhancedVAE(data_dim=self.data_dim, latent_dim=4, learning_rate=0.0005, seed=self.seed)
        
        # Initial imputation
        mean_imputer = SimpleImputer(strategy='mean')
        mean_imputer.fit(data)
        initial_imputed = mean_imputer.transform(data)
        data_filled = np.where(mask == 1, data, initial_imputed)
        
        vae.fit(data_filled, mask, epochs=500, batch_size=64)
        self.models.append(('vae', vae))
        
        # Set feature-specific weights for H and V
        for i, col_name in enumerate(column_names):
            if 'H' in col_name:
                # Weights for H
                self.feature_weights[i] = {
                    'knn': 0.15,
                    'gb_mice': 0.20,
                    'rf_mice': 0.20,
                    'et_mice': 0.15,
                    'feature_deep': 0.20,
                    'vae': 0.10
                }
            elif 'V' in col_name:
                # Weights for V
                self.feature_weights[i] = {
                    'knn': 0.15,
                    'gb_mice': 0.25,
                    'rf_mice': 0.25,
                    'et_mice': 0.20,
                    'feature_deep': 0.10,
                    'vae': 0.05
                }
            else:
                # Default weights for other features
                self.feature_weights[i] = {
                    'knn': 0.17,
                    'gb_mice': 0.17,
                    'rf_mice': 0.17,
                    'et_mice': 0.17,
                    'feature_deep': 0.16,
                    'vae': 0.16
                }
        
        print("Ensemble training complete")
    
    def impute(self, data, mask):
        """Predict using the enhanced ensemble."""
        n_samples, n_features = data.shape
        imputed = np.zeros_like(data)
        
        # Get predictions from all models
        all_predictions = {}
        for name, model in self.models:
            if name == 'vae':
                mean_imputer = SimpleImputer(strategy='mean')
                mean_imputer.fit(data)
                initial_imputed = mean_imputer.transform(data)
                data_filled = np.where(mask == 1, data, initial_imputed)
                pred = model.impute(data_filled, mask)
            elif name == 'feature_deep':
                pred = model.impute(data, mask)
            else:
                pred = model.transform(data)
            all_predictions[name] = pred
        
        # Take a weighted average using each feature's weights
        for feature_idx in range(n_features):
            if feature_idx in self.feature_weights:
                weights = self.feature_weights[feature_idx]
                feature_pred = np.zeros(n_samples)
                
                for model_name, weight in weights.items():
                    if model_name in all_predictions:
                        feature_pred += all_predictions[model_name][:, feature_idx] * weight
                
                imputed[:, feature_idx] = feature_pred
            else:
                # Use the mean when feature-specific weights are unavailable
                feature_pred = np.zeros(n_samples)
                n_models = len(all_predictions)
                for pred in all_predictions.values():
                    feature_pred += pred[:, feature_idx] / n_models
                imputed[:, feature_idx] = feature_pred
        
        # Preserve observed values
        imputed = data * mask + imputed * (1 - mask)
        
        return imputed

# 7. Evaluation
def evaluate_imputation_original_scale(model, data_array, mask, column_names, 
                                      data_raw, scalers, transform_info):
    """Evaluate the model in the original units using cross-validation."""
    print("\n=== Model performance in the original units ===")
    
    results = []
    all_observations = []
    
    original_index = data_raw.index
    
    for col_idx, col_name in enumerate(column_names):
        obs_mask = mask[:, col_idx] == 1
        obs_indices = np.where(obs_mask)[0]
        
        if len(obs_indices) < 100:
            continue
        
        # Use five-fold cross-validation
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        
        all_true = []
        all_pred = []
        all_indices = []
        
        for train_idx, test_idx in kf.split(obs_indices):
            test_indices = obs_indices[test_idx]
            
            # Create the test mask
            test_mask = mask.copy()
            test_mask[test_indices, col_idx] = 0
            
            # Impute on the scaled data
            imputed_normalized = model.impute(data_array, test_mask)
            
            # Convert back to the original units
            imputed_original = inverse_transform_array(
                imputed_normalized, scalers, transform_info, 
                column_names, original_index
            )
            
            # Get the observed values in the original units
            true_values_original = data_raw.iloc[test_indices][col_name].values
            pred_values_original = imputed_original[col_name].iloc[test_indices].values
            
            all_true.extend(true_values_original)
            all_pred.extend(pred_values_original)
            all_indices.extend(test_indices)
            
            # Record each prediction
            for idx, true_val, pred_val in zip(test_indices, true_values_original, pred_values_original):
                all_observations.append({
                    'Row Index': idx,
                    'Feature': col_name,
                    'Observed Value': true_val,
                    'Predicted Value': pred_val,
                    'Absolute Error': abs(true_val - pred_val)
                })
        
        all_true = np.array(all_true)
        all_pred = np.array(all_pred)
        
        # Calculate metrics in the original units
        mae = mean_absolute_error(all_true, all_pred)
        rmse = np.sqrt(mean_squared_error(all_true, all_pred))
        
        # Calculate R²
        ss_res = np.sum((all_true - all_pred) ** 2)
        ss_tot = np.sum((all_true - np.mean(all_true)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        # Correlation coefficient
        if np.std(all_true) > 0 and np.std(all_pred) > 0:
            correlation = np.corrcoef(all_true, all_pred)[0, 1]
        else:
            correlation = 0
        
        results.append({
            'Feature': col_name,
            'R² (Original Scale)': r2,
            'Correlation': correlation,
            'MAE': mae,
            'RMSE': rmse,
            'Sample Count': len(all_true)
        })
        
        print(f"{col_name}: R²={r2:.4f}, correlation={correlation:.4f}, MAE={mae:.4f}")
    
    return pd.DataFrame(results), pd.DataFrame(all_observations)

# 8. Main function
def main():
    """Run the workflow."""
    print("=" * 80)
    print("VAE imputation for H and V")
    print("H: Box-Cox transform and IQR clipping")
    print("V: quantile transform")
    print("=" * 80)
    
    try:
        set_all_seeds(SEED)
        
        # 1. Load the data
        print("\n1. Loading data...")
        data_raw, original_data = load_and_preprocess_data()
        
        # Keep the original data for evaluation
        data_raw_copy = data_raw.copy()
        
        # 2. Treat outliers
        print("\n2. Treating outliers...")
        data_clean, transform_info = enhanced_outlier_handling(data_raw)
        
        # 3. Scale the data
        print("\n3. Scaling the data...")
        data_normalized, scalers = enhanced_normalization(data_clean)
        
        # Create the mask
        mask = (~data_normalized.isnull()).astype(np.float32).values
        data_array = data_normalized.fillna(0.5).values.astype(np.float32)
        
        print(f"\nData shape: {data_array.shape}")
        print(f"Missing rate: {(1 - mask.mean())*100:.2f}%")
        
        # 4. Train the ensemble
        print("\n4. Training the ensemble...")
        model = EnhancedEnsemble(data_dim=data_array.shape[1], seed=SEED)
        model.fit(data_array, mask, data_normalized.columns)
        
        # 5. Evaluate in the original units
        print("\n5. Evaluating the model in the original units...")
        eval_results, observations_df = evaluate_imputation_original_scale(
            model, data_array, mask, data_normalized.columns, 
            data_raw_copy, scalers, transform_info
        )
        
        # 6. Final imputation
        print("\n6. Generating the final imputed data...")
        final_imputed_normalized = model.impute(data_array, mask)
        
        # Convert back to the original units
        final_imputed_original = inverse_transform_array(
            final_imputed_normalized, scalers, transform_info, 
            data_normalized.columns, data_raw_copy.index
        )
        
        # Assemble the completed data
        final_result = data_raw_copy.astype(np.float64)
        for col in final_result.columns:
            missing_mask = data_raw_copy[col].isna()
            final_result.loc[missing_mask, col] = final_imputed_original.loc[missing_mask, col]
        
        # 7. Report the results
        print("\n" + "=" * 80)
        print("Model performance summary (original scale)")
        print("=" * 80)
        
        for _, row in eval_results.iterrows():
            print(f"\n{row['Feature']}:")
            print(f"  R² (original scale): {row['R² (Original Scale)']:.4f}")
            print(f"  Correlation: {row['Correlation']:.4f}")
            print(f"  MAE: {row['MAE']:.4f}")
            print(f"  RMSE: {row['RMSE']:.4f}")
            print(f"  Evaluation samples: {row['Sample Count']}")
        
        print(f"\nMean R² (original scale): {eval_results['R² (Original Scale)'].mean():.4f}")
        print(f"Minimum R² (original scale): {eval_results['R² (Original Scale)'].min():.4f}")
        
        # Save the results
        observations_df.to_csv("observed_vs_predicted_others.csv", index=False, encoding='utf-8-sig')
        final_result.to_csv("imputed_others.csv", index=False, encoding='utf-8-sig')
        eval_results.to_csv("evaluation_others.csv", index=False, encoding='utf-8-sig')
        
        print("\nFiles saved:")
        print("1. observed_vs_predicted_others.csv")
        print("2. imputed_others.csv")
        print("3. evaluation_others.csv")
        
        # Check model performance
        print("\n" + "=" * 80)
        print("Model performance check")
        print("=" * 80)
        
        target_r2 = 0.80  # R² target
        
        for _, row in eval_results.iterrows():
            feature = row['Feature']
            r2 = row['R² (Original Scale)']
            if r2 >= target_r2:
                print(f"✅ {feature}: R²={r2:.4f} - target met")
            elif r2 >= 0.70:
                print(f"⚠️ {feature}: R²={r2:.4f} - near the target")
            else:
                print(f"❌ {feature}: R²={r2:.4f} - needs further improvement")
        
        return final_result, eval_results, observations_df
        
    except Exception as e:
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None

if __name__ == "__main__":
    print("\nStarting the workflow...")
    print("Data processing methods:")
    print("1. H: Box-Cox transform with IQR clipping at a multiplier of 3")
    print("2. V: quantile transform to a standard normal distribution")
    print("3. VAE architecture designed for two features")
    print("4. Ensemble of six imputation models")
    
    start_time = time.time()
    result, evaluation, observations = main()
    
    if result is not None:
        print(f"\n✅ Run complete!")
        print(f"Elapsed time: {(time.time() - start_time)/60:.2f} minutes")
    else:
        print("\n❌ Run failed")
    
    gc.collect()

TensorFlow version: 2.21.0
No GPU found. Running on the CPU.


Starting the workflow...
Data processing methods:
1. H: Box-Cox transform with IQR clipping at a multiplier of 3
2. V: quantile transform to a standard normal distribution
3. VAE architecture designed for two features
4. Ensemble of six imputation models
VAE imputation for H and V
H: Box-Cox transform and IQR clipping
V: quantile transform

1. Loading data...

Error: [Errno 2] No such file or directory: './Others.csv'

❌ Run failed


Traceback (most recent call last):
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_5900\2079810894.py", line 849, in main
    data_raw, original_data = load_and_preprocess_data()
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_5900\2079810894.py", line 70, in load_and_preprocess_data
    data = pd.read_csv("./Others.csv", low_memory=True)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 873, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 300, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 1645, i

In [2]:
import pandas as pd

def add_L_column_to_results():
    """
    Add the L(m) column to the imputed results.
    """
    # 1. Read the source data to retrieve L(m)
    original_data = pd.read_csv("./Others.csv", low_memory=True)
    
    # 2. Check whether L(m) is present
    if 'L(m)' not in original_data.columns:
        print("The source data does not contain L(m)")
        return
    
    # 3. Extract L(m)
    L_column = original_data['L(m)'].copy()
    print(f"Extracted L(m) with {len(L_column)} rows")
    
    # 4. Read the imputed results
    try:
        imputed_results = pd.read_csv("imputed_others.csv")
        print(f"Imputed data shape: {imputed_results.shape}")
    except FileNotFoundError:
        print("Imputed results file not found. Run the imputation cell first.")
        return
    
    # 5. Check that the row counts match
    if len(imputed_results) != len(L_column):
        print(f"Warning: imputed results have {len(imputed_results)} rows, while L(m) has {len(L_column)} rows")
        # If row counts differ, align both tables to the shorter length
        min_rows = min(len(imputed_results), len(L_column))
        imputed_results = imputed_results.iloc[:min_rows]
        L_column = L_column.iloc[:min_rows]
    
    # 6. Add L(m) to the results
    imputed_results['L(m)'] = L_column.values
    
    # 7. Save the updated results
    output_filename = "imputed_others_with_runout.csv"
    imputed_results.to_csv(output_filename, index=False)
    
    print(f"\nAdded L(m) to the imputed results.")
    print(f"Saved the new file as: {output_filename}")
    print(f"Final data shape: {imputed_results.shape}")
    print(f"Column names: {list(imputed_results.columns)}")
    
    # 8. Report summary statistics for L(m)
    print("\nL(m) summary statistics:")
    print(f"  Data type: {L_column.dtype}")
    print(f"  Unique values: {L_column.nunique()}")
    print(f"  Missing values: {L_column.isna().sum()}")
    if pd.api.types.is_numeric_dtype(L_column):
        print(f"  Minimum: {L_column.min()}")
        print(f"  Maximum: {L_column.max()}")
        print(f"  Mean: {L_column.mean():.2f}")
    else:
        print(f"  First five values: {L_column.head().tolist()}")
    
    return imputed_results

# Run the function
if __name__ == "__main__":
    result_with_L = add_L_column_to_results()

Extracted L(m) with 579 rows
Imputed data shape: (579, 2)

Added L(m) to the imputed results.
Saved the new file as: imputed_others_with_runout.csv
Final data shape: (579, 3)
Column names: ['H(m)', 'V(m³)', 'L(m)']

L(m) summary statistics:
  Data type: float64
  Unique values: 432
  Missing values: 0
  Minimum: 1.17
  Maximum: 1610000.0
  Mean: 7553.59
